In [1]:
import os
import subprocess
from datasets import load_dataset
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm


In [2]:
# ================= CONFIG =================
FULL_AUDIO_DIR = "musiccaps/full_audio"
SEG_AUDIO_DIR = "musiccaps/segments"
FAILED_LOG = "musiccaps/failed.txt"
MAX_WORKERS = 12

os.makedirs(FULL_AUDIO_DIR, exist_ok=True)
os.makedirs(SEG_AUDIO_DIR, exist_ok=True)


In [3]:
# ================= DATA =================
ds = load_dataset("google/musiccaps", split="train")


In [4]:
# ================= HELPERS =================
def download_audio(ytid):
    out = f"{FULL_AUDIO_DIR}/{ytid}.m4a"
    if os.path.exists(out):
        return out

    url = f"https://www.youtube.com/watch?v={ytid}"

    res = subprocess.run(
        [
            "yt-dlp",
            "--sleep-interval", "1",
            "--throttled-rate", "100K",
            "--concurrent-fragments", "1",
            "-f", "bestaudio[ext=m4a]/bestaudio",
            url,
            "-o", out
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    return out if res.returncode == 0 and os.path.exists(out) else None

    
def cut_segment(src, start, end, out):
    if not os.path.exists(src):
        return False
    
    result = subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-ss", str(start),
            "-to", str(end),
            "-i", src,
            "-c:a", "pcm_s16le",
            "-ar", "44100",
            "-f", "wav",
            out
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        text=True
    )
    
    if result.returncode != 0:
        print(f"FFmpeg error for {src}: {result.stderr}")
    
    return result.returncode == 0 and os.path.exists(out)


def process_sample(sample):
    ytid = sample["ytid"]
    start = sample["start_s"]
    end = sample["end_s"]

    out = f"{SEG_AUDIO_DIR}/{ytid}_{start:.2f}_{end:.2f}.wav"
    if os.path.exists(out):
        return

    full = download_audio(ytid)
    if full is None:
        with open(FAILED_LOG, "a") as f:
            f.write(f"DOWNLOAD_FAIL {ytid}\n")
        return

    if not cut_segment(full, start, end, out):
        with open(FAILED_LOG, "a") as f:
            f.write(f"CUT_FAIL {ytid} {start} {end}\n")


In [ ]:
# ================= RUN =================
NUM_SAMPLES = len(ds)

# Random subset
ds_subset = ds.shuffle(seed=42).select(range(NUM_SAMPLES))

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    list(tqdm(pool.map(process_sample, ds_subset), total=NUM_SAMPLES))

print("Done.")

 67%|██████▋   | 3720/5521 [19:19<26:09,  1.15it/s]  

In [ ]:
from datasets import Audio
import soundfile as sf

# Filter first
def has_segment(sample):
    ytid = sample['ytid']
    start = sample['start_s']
    end = sample['end_s']
    path = f"{SEG_AUDIO_DIR}/{ytid}_{start:.2f}_{end:.2f}.wav"
    return os.path.exists(path)

ds_filtered = ds.filter(has_segment, desc="Filtering", load_from_cache_file=False,)
total = len(ds_filtered)
print(f"Found {total} samples with segments")

# Just add the path as a string - no audio loading yet
def add_audio_path(sample):
    ytid = sample['ytid']
    start = sample['start_s']
    end = sample['end_s']
    path = f"{SEG_AUDIO_DIR}/{ytid}_{start:.2f}_{end:.2f}.wav"
    sample["audio"] = path  # Store as string path
    return sample

# This is fast - just adds paths
ds_with_paths = ds_filtered.map(add_audio_path, desc="Adding paths", num_proc=0, load_from_cache_file=False,)

# Cast to Audio feature - this handles loading and serialization properly
ds_with_audio = ds_with_paths.cast_column("audio", Audio(sampling_rate=44100))

# Save to disk
ds_with_audio.save_to_disk("musiccaps/dataset_audio")
print("Done! Dataset saved with Audio feature")

Filtering:   0%|          | 0/5521 [00:00<?, ? examples/s]

Found 964 samples with segments


Adding paths:   0%|          | 0/964 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/964 [00:00<?, ? examples/s]

Done! Dataset saved with Audio feature


In [7]:
ds[0]

{'ytid': '-0Gj8-vB1q4',
 'start_s': 30,
 'end_s': 40,
 'audioset_positive_labels': '/m/0140xf,/m/02cjck,/m/04rlf',
 'aspect_list': "['low quality', 'sustained strings melody', 'soft female vocal', 'mellow piano melody', 'sad', 'soulful', 'ballad']",
 'caption': 'The low quality recording features a ballad song that contains sustained strings, mellow piano melody and soft female vocal singing over it. It sounds sad and soulful, like something you would hear at Sunday services.',
 'author_id': 4,
 'is_balanced_subset': False,
 'is_audioset_eval': True}

In [5]:
import gradio as gr
from datasets import load_from_disk

ds = load_from_disk("musiccaps/dataset_audio")
print(len(ds))
ex = ds[0]
print(type(ex['audio']))

964
<class 'datasets.features._torchcodec.AudioDecoder'>
